# 3. Ingeniería de Características (Feature Engineering)

Este notebook toma como entrada `data/titanic_clean.csv` (salida de `EDA.ipynb` / `EDA_estocastico.ipynb`, ya sin nulos y con la variable `titulo` extraída del nombre) y aplica el **enfoque moderno de scikit-learn** para codificar variables categóricas y escalar variables numéricas: `ColumnTransformer` + `Pipeline`, en vez de mapeos manuales de pandas.

## ¿Por qué no mapear a mano?

El enfoque tradicional —por ejemplo, `df['sex'].map({'male': 0, 'female': 1})` o `pd.get_dummies(df, columns=['embarked'])` guardado en un csv intermedio— tiene tres problemas serios en producción:

1. **Fuga de información (data leakage)**: si se escala (`StandardScaler`) o se calculan estadísticos *antes* de separar train/test, el modelo "ve" información del set de prueba durante el ajuste.
2. **Inconsistencia entre entrenamiento y predicción**: si el mapeo se escribe a mano en un notebook y en la API de predicción por separado, un cambio en una de las dos copias rompe la otra silenciosamente.
3. **Categorías no vistas**: un mapeo manual explota si llega una categoría nueva (p. ej. un valor de `embarked` no contemplado); `OneHotEncoder(handle_unknown='ignore')` la ignora de forma segura.

## Hallazgo importante sobre `data/titanic_normalized.csv`

En el repositorio ya existe un archivo `data/titanic_normalized.csv` con las variables categóricas codificadas en one-hot (con la primera categoría descartada, p. ej. existe `sex_male` pero no `sex_female`) y las numéricas escaladas. **Sin embargo, no hay ningún notebook ni script en este proyecto que genere ese archivo** — ni `OneHotEncoder` de scikit-learn ni `pd.get_dummies` persistente en ningún lugar del código. Es decir, el **OneHotEncoder real y reproducible todavía no se había hecho**; `titanic_normalized.csv` es un artefacto huérfano.

Este notebook resuelve eso: construye la codificación desde cero, de forma reproducible, a partir de `titanic_clean.csv`, y guarda el resultado en un archivo nuevo — `data/titanic_feature.csv` — sin tocar el `titanic_normalized.csv` existente.

In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

df = pd.read_csv('data/titanic_clean.csv')
print(df.shape)
print(df.dtypes)
df.head()

(891, 11)
passenger_id      int64
survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
familiares        int64
fare            float64
embarked         object
titulo           object
dtype: object


,passenger_id,survived,pclass,sex,age,sibsp,parch,familiares,fare,embarked,titulo
0,1,0,3,male,22.0,1,0,1,7.2500,S,Mr
1,2,1,1,female,38.0,1,0,1,71.2833,C,Mrs
2,3,1,3,female,26.0,0,0,0,7.9250,S,Miss
3,4,1,1,female,35.0,1,0,1,53.1000,S,Mrs
4,5,0,3,male,35.0,0,0,0,8.0500,S,Mr


Confirmamos que no queden nulos (deberían haberse resuelto en la etapa de limpieza/imputación).

In [3]:
df.isnull().sum()

passenger_id    0
survived        0
pclass          0
sex             0
age             0
sibsp           0
parch           0
familiares      0
fare            0
embarked        0
titulo          0
dtype: int64

## Definición de columnas

- **Numéricas** (se escalan con `StandardScaler`, media 0 / desviación 1): `age`, `sibsp`, `parch`, `familiares`.
- **`fare`** se trata aparte: su distribución está muy sesgada a la derecha (pocos pasajeros pagaron tarifas muy altas), así que primero se aplica `log1p` (logaritmo de `1 + fare`, para poder manejar valores en 0) y luego se escala. Esto reduce la asimetría sin perder la información.
- **Categóricas** (se codifican con `OneHotEncoder`): `sex`, `embarked`, `titulo`, `pclass`. Tratamos `pclass` como categórica (no ordinal) porque no hay razón para asumir que la diferencia entre 1ª y 2ª clase es "igual" a la diferencia entre 2ª y 3ª en su efecto sobre la supervivencia.
- **`passenger_id`** se descarta: es un identificador, no aporta información predictiva.
- **`survived`** es la variable objetivo (target), no se transforma.

In [4]:
numeric_cols = ['age', 'sibsp', 'parch', 'familiares']
fare_col = ['fare']
categorical_cols = ['sex', 'embarked', 'titulo', 'pclass']
target_col = 'survived'

X = df[numeric_cols + fare_col + categorical_cols]
y = df[target_col]

## `ColumnTransformer`: por qué `drop='first'`

En `EDA_estocastico.ipynb` ya se calculó el VIF (Variance Inflation Factor) y se encontró multicolinealidad severa (VIF > 30) al mantener todas las categorías dummy de `sex` y `titulo` a la vez — el clásico problema de la **trampa de las variables dummy** (dummy variable trap): si conoces todas las categorías de una variable menos una, la última queda determinada por las demás.

Por eso aquí usamos `OneHotEncoder(drop='first', handle_unknown='ignore')`: se descarta una categoría de referencia por variable (esa categoría queda representada implícitamente cuando todas las demás columnas dummy son 0), evitando la redundancia y la multicolinealidad, y a la vez se ignoran con seguridad categorías no vistas en producción.

In [5]:
fare_pipeline = Pipeline(steps=[
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scale', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('fare', fare_pipeline, fare_col),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
])

preprocessor

,transformers,"[('num', ...), ('fare', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


## Ajuste y transformación

Aquí ajustamos (`fit_transform`) el preprocesador sobre **todo** el dataset, porque el objetivo de este notebook es producir un artefacto de features para exploración y para alimentar el futuro notebook de entrenamiento — no es todavía el pipeline final de modelado.

> **Importante para el notebook de entrenamiento (`train.ipynb`, futuro):** ahí sí debe hacerse `train_test_split` primero y luego `fit` del `ColumnTransformer` **únicamente sobre el set de entrenamiento** (usando un `Pipeline` que incluya el estimador), para no filtrar información del set de prueba — el mismo criterio que ya aplicaron con la variable objetivo en la imputación estocástica de `age` (reglas de Rubin / van Buuren).

In [6]:
X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

print('Registro original (primera fila):')
print(X.iloc[0])
print('\nRegistro transformado (primera fila):')
for name, value in zip(feature_names, X_transformed[0]):
    print(f'  {name}: {value:.4f}')

Registro original (primera fila):
age           22.0
sibsp            1
parch            0
familiares       1
fare          7.25
sex           male
embarked         S
titulo          Mr
pclass           3
Name: 0, dtype: object

Registro transformado (primera fila):
  num__age: -0.5578
  num__sibsp: 0.4328
  num__parch: -0.4737
  num__familiares: 0.0592
  fare__fare: -0.8797
  cat__sex_male: 1.0000
  cat__embarked_Q: 0.0000
  cat__embarked_S: 1.0000
  cat__titulo_Miss: 0.0000
  cat__titulo_Mr: 1.0000
  cat__titulo_Mrs: 0.0000
  cat__titulo_Otro: 0.0000
  cat__pclass_2: 0.0000
  cat__pclass_3: 1.0000


In [7]:
df_features = pd.DataFrame(X_transformed, columns=feature_names)
df_features.insert(0, 'survived', y.values)
df_features.head()

,survived,num__age,num__sibsp,num__parch,num__familiares,fare__fare,cat__sex_male,cat__embarked_Q,cat__embarked_S,cat__titulo_Miss,cat__titulo_Mr,cat__titulo_Mrs,cat__titulo_Otro,cat__pclass_2,cat__pclass_3
0,0,-0.557791,0.432793,-0.473674,0.059160,-0.879741,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1,0.574173,0.432793,-0.473674,0.059160,1.361220,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,1,-0.274800,-0.474545,-0.473674,-0.560975,-0.798540,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3,1,0.361930,0.432793,-0.473674,0.059160,1.062038,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0,0.361930,-0.474545,-0.473674,-0.560975,-0.784179,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [8]:
df_features.isnull().sum()

survived            0
num__age            0
num__sibsp          0
num__parch          0
num__familiares     0
fare__fare          0
cat__sex_male       0
cat__embarked_Q     0
cat__embarked_S     0
cat__titulo_Miss    0
cat__titulo_Mr      0
cat__titulo_Mrs     0
cat__titulo_Otro    0
cat__pclass_2       0
cat__pclass_3       0
dtype: int64

## Guardado

Guardamos el resultado en `data/titanic_feature.csv` — un archivo nuevo, distinto de `data/titanic_normalized.csv` (que se deja intacto, ya que no era reproducible).

In [9]:
df_features.to_csv('data/titanic_feature.csv', index=False)
print('Guardado en data/titanic_feature.csv:', df_features.shape)

Guardado en data/titanic_feature.csv: (891, 15)


## Resumen

- Se construyó un `ColumnTransformer` reproducible con `StandardScaler` (numéricas), `log1p + StandardScaler` (`fare`) y `OneHotEncoder(drop='first', handle_unknown='ignore')` (categóricas: `sex`, `embarked`, `titulo`, `pclass`).
- `data/titanic_normalized.csv` (existente) estaba codificado de forma similar (one-hot con `drop_first`), pero **ningún código del repositorio lo generaba** — era un artefacto huérfano. Este notebook es ahora la fuente reproducible de esa codificación.
- El resultado se guardó en `data/titanic_feature.csv`.
- Pendiente para el notebook de entrenamiento: reajustar este mismo `ColumnTransformer` solo sobre el split de entrenamiento (dentro de un `Pipeline` junto con el modelo) para evitar data leakage.